# 06 时间验证与结果汇总

这本 Notebook 在同一批时间验证患者上比较 ResNet、临床、ResNet 联合临床、影像组学，以及影像组学联合临床五组模型，汇总预测概率、分类指标、ROC 和校准图。

模型、数据处理参数和分类阈值均来自前面的开发阶段。完成 02 的模型保存后即可验证三组分类基线；加入影像组学需要先完成 01，加入自动掩模实验还需要 05 的验证预测。

下面先读取配置，定位本次实验的模型和结果文件。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg


def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics", "explain": root / "explain", "segmentation": root / "segmentation",
        "reports": root / "reports",
    }


def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result


def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "original_resnet_integration_no_architecture_optimization", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)


def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table


def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)


def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)


def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")
    if manifest["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("Notebook 定义与数据包导出时不同，请使用对应版本或重新导出。")


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本数据包只有分类缓存，没有原分辨率影像；影像组学和分割需另外准备原生数据。")


In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


## 模型推理所需的工具

这里加载 PyTorch、ResNet50 和相关工具，供后面构建网络并恢复训练参数。`DEVICE` 决定模型及影像张量的计算位置，预测函数按照设定的批大小分批处理患者。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 分类网络与概率预测

`MaskEncoder` 和 `DualBranchModel` 定义训练时使用的双分支结构：每层 MRI 与病灶特征拼接，再通过注意力汇总四层信息。`predict_proba` 将模型切换到评估状态，关闭梯度计算，分批返回阳性概率。

这里恢复网络结构后，将加载 02 保存的最终开发模型参数。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: DualBranchModel
class DualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


# 直接复制：code/train_resnet50_manual_cv.py :: predict_proba
@torch.no_grad()
def predict_proba(
    model: nn.Module,
    X: torch.Tensor,
    M: torch.Tensor,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size].to(DEVICE)
        mb = M[i : i + batch_size].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outs.append(torch.sigmoid(model(xb, mb)).float().cpu().numpy())
    return np.concatenate(outs)


## 核对模型版本并预测三组分类基线

`verify_lock` 对照开发阶段保存的校验信息，核对模型、阈值、队列、缓存和代码定义。`load_original_model` 随后载入最终 ResNet 权重。

`predict_original_validation` 从患者清单中取出时间验证病例，计算 ResNet 概率，再分别得到临床模型和 ResNet 联合临床模型的概率。分类标签使用各模型在开发阶段确定的阈值，并与 MR 编号、真实标签一起整理成表。

如果传入自动掩模缓存，函数会先核对患者顺序和数组形状，再运行相同分类模型，给结果加上自动掩模替换的标记。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p


def verify_lock(cfg):
    p = paths(cfg)
    metadata = json.loads((p["temporal"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if digest(p["cache"]) != metadata["cache_sha256"]:
        raise ValueError("影像缓存与开发锁定版本不一致。")
    for name, expected in metadata["artifacts"].items():
        if digest(p["run"] / name) != expected:
            raise ValueError(f"固定产物发生变化：{name}")
    if source_snapshot(cfg) != metadata["notebook_definitions_sha256"]:
        raise ValueError("Notebook中的函数或模型定义发生变化，请核对对应的实验版本。")


def load_original_model(cfg, checkpoint=None):
    import torch
    configure(cfg)
    model = DualBranchModel().to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["temporal"] / "final_development_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model


def predict_original_validation(cfg, automatic_masks=None):
    import joblib
    import torch
    verify_lock(cfg)
    p = configure(cfg)
    val = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    idx = val.cache_index.to_numpy(dtype=int)
    with np.load(p["cache"], allow_pickle=False) as cache:
        X = torch.from_numpy(cache["images"][idx].astype(np.float32))
        M = torch.from_numpy(cache["masks"][idx].astype(np.uint8))
    suffix = ""
    if automatic_masks is not None:
        with np.load(automatic_masks, allow_pickle=False) as predicted:
            if predicted["mrs"].astype(str).tolist() != val.mr.tolist() or predicted["masks"].shape != tuple(M.shape):
                raise ValueError("自动掩模患者顺序或形状不匹配。")
            M = torch.from_numpy(predicted["masks"].astype(np.uint8))
        suffix = "_AutoMask_Substitution"
    probability = predict_proba(load_original_model(cfg), X, M)
    dl_metrics = json.loads((p["temporal"] / "development_cv_metrics.json").read_text(encoding="utf-8"))
    meta = json.loads((p["temporal"] / "clinical_fusion_meta.json").read_text(encoding="utf-8"))
    models = joblib.load(p["temporal"] / "clinical_fusion.joblib")
    clinical = val[CLINICAL].to_numpy(dtype=np.float32)
    probabilities = {"ResNet50" + suffix: (probability, dl_metrics["youden_threshold"]),
        "ResNet50_Clinical5" + suffix: (models["ResNet50_Clinical5"].predict_proba(np.column_stack([probability, clinical]))[:, 1], meta["ResNet50_Clinical5"]["threshold"])}
    if not suffix:
        probabilities["Clinical5"] = (models["Clinical5"].predict_proba(clinical)[:, 1], meta["Clinical5"]["threshold"])
    predictions, summaries = [], []
    for name, (prob, threshold) in probabilities.items():
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "manual_model_auto_mask_substitution" if suffix else "original_baseline"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)


## 用已保存的影像组学模型预测

`predict_radiomics_validation` 核对特征表和模型文件，按 MR 编号将时间验证患者与特征一一匹配，然后加载 01 保存的处理流程和逻辑回归模型。

缺失值填补、特征选择和标准化均使用开发阶段学到的参数，分类阈值也从已保存的记录读取。输出格式与 ResNet 组一致，便于合并比较。


In [ ]:
from __future__ import annotations

def predict_radiomics_validation(cfg):
    import joblib
    p, cohort = paths(cfg), read_cohort(cfg)
    lock = json.loads((p["radiomics"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_sha256"] != digest(p["cohort"]):
        raise ValueError("固定队列发生变化。")
    for name, expected in lock["artifacts"].items():
        if digest(p["radiomics"] / name) != expected:
            raise ValueError(f"影像组学固定产物发生变化：{name}")
    feature_table = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    val = cohort.query("split == 'validation'").merge(feature_table.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    models = joblib.load(p["radiomics"] / "models.joblib")
    meta = json.loads((p["radiomics"] / "model_meta.json").read_text(encoding="utf-8"))
    predictions, summaries = [], []
    for name, model in models.items():
        prob = model.predict_proba(val[meta[name]["features"]].replace([np.inf, -np.inf], np.nan))[:, 1]
        threshold = meta[name]["threshold"]
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "radiomics_full_volume_nested_feature_selection"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)


## 评价自动勾画，并转换为分类输入

`evaluate_masks` 在原生影像空间中逐患者、逐标签比较自动与人工掩模，计算 Dice 和以毫米为单位的 HD95。没有可比较前景的情况会单独记录状态，汇总时同时显示有效病例数。

`build_validation_mask_cache` 按已有取层记录截取四层，拆分三类病灶，并用最近邻插值缩放到 224 × 224。`verify_validation_mask_cache` 核对转换后的缓存与原生预测文件是否属于同一次结果，确保分割评价和分类实验使用相同的掩模。


In [ ]:
from __future__ import annotations

def locations(cfg):
    root = paths(cfg)["segmentation"]
    dataset_id = int(cfg["segmentation"]["dataset_id"])
    if not 100 <= dataset_id <= 999:
        raise ValueError("本项目使用100～999的分割数据集ID。")
    name = f"Dataset{dataset_id:03d}_PSCI"
    return {"root": root, "name": name, "raw": root / "nnUNet_raw" / name,
            "preprocessed": root / "nnUNet_preprocessed" / name,
            "results": root / "nnUNet_results", "validation": root / "validation_predictions",
            "oof": root / "development_oof_predictions"}


def _volumes(patient, predicted_path=None):
    import nibabel as nib
    image = nib.load(str(patient.t1))
    truth = nib.load(str(patient.manual_mask))
    others = [nib.load(str(patient[s])) for s in SEQUENCES[1:]] + [truth]
    if predicted_path is not None:
        others.append(nib.load(str(predicted_path)))
    if any(v.shape != image.shape or not np.allclose(v.affine, image.affine, atol=1e-4) for v in others):
        raise ValueError(f"{patient.mr}的影像、真值或预测几何不一致；不自动重采样修正。")
    for volume in others[-2:] if predicted_path is not None else [truth]:
        values = np.unique(np.asarray(volume.dataobj))
        if not np.isfinite(values).all() or not set(values) <= {0, 1, 2, 3}:
            raise ValueError(f"{patient.mr}标签不是0/1/2/3。")
    prediction = others[-1] if predicted_path is not None else None
    return image, truth, prediction


def verify_dataset(cfg):
    loc, cohort = locations(cfg), read_cohort(cfg)
    meta = json.loads((loc["root"] / "dataset_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_sha256"] != digest(paths(cfg)["cohort"]):
        raise ValueError("分割导出后病例清单发生变化。")
    if meta["configuration"] != cfg["segmentation"]["configuration"] or meta["label_names"] != cfg["label_names"]:
        raise ValueError("分割配置或标签定义发生变化，请使用新的run_id导出。")
    expected = set(cohort.loc[cohort.split == "development", "mr"])
    actual = {p.name.removesuffix(".nii.gz") for p in (loc["raw"] / "labelsTr").glob("*.nii.gz")}
    if actual != expected:
        raise ValueError("labelsTr与开发队列不完全一致，禁止继续训练。")


def evaluate_masks(cfg, scope="validation"):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    verify_dataset(cfg)
    loc, cohort = locations(cfg), read_cohort(cfg)
    if scope not in ("validation", "development_oof"):
        raise ValueError("scope只能是validation或development_oof。")
    cases = cohort.query("split == 'validation'" if scope == "validation" else "split == 'development'")
    rows = []
    for _, patient in cases.iterrows():
        directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(patient.development_fold) - 1}"
        image, truth, prediction = _volumes(patient, directory / f"{patient.mr}.nii.gz")
        if image.header.get_xyzt_units()[0] != "mm":
            raise ValueError(f"{patient.mr}的NIfTI空间单位不是明确的mm；请核查后再报告HD95_mm。")
        true_values, pred_values = np.asarray(truth.dataobj), np.asarray(prediction.dataobj)
        for label in (1, 2, 3):
            a, b = true_values == label, pred_values == label
            na, nb = int(a.sum()), int(b.sum())
            row = {"mr": patient.mr, "scope": scope, "label": label, "true_voxels": na, "predicted_voxels": nb}
            if not na and not nb:
                row.update(dice=np.nan, hd95_mm=np.nan, status="both_empty_excluded_from_mean")
            elif not na or not nb:
                row.update(dice=0.0, hd95_mm=np.nan, status="one_empty_hd95_undefined")
            else:
                surface_a = a ^ binary_erosion(a, border_value=0)
                surface_b = b ^ binary_erosion(b, border_value=0)
                distances = np.concatenate([distance_transform_edt(~surface_b, sampling=image.header.get_zooms()[:3])[surface_a],
                                            distance_transform_edt(~surface_a, sampling=image.header.get_zooms()[:3])[surface_b]])
                row.update(dice=float(2 * (a & b).sum() / (na + nb)), hd95_mm=float(np.percentile(distances, 95)), status="ok")
            rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(loc["root"] / f"{scope}_segmentation_metrics.csv", index=False)
    summary = table.groupby("label").agg(n_cases=("mr", "size"), n_dice_defined=("dice", "count"),
        mean_dice=("dice", "mean"), n_hd95_defined=("hd95_mm", "count"), mean_hd95_mm=("hd95_mm", "mean"))
    summary.to_csv(loc["root"] / f"{scope}_segmentation_summary.csv")
    return table, summary


def build_validation_mask_cache(cfg):
    from scipy.ndimage import zoom
    verify_dataset(cfg)
    loc, p = locations(cfg), paths(cfg)
    target = loc["root"] / "validation_masks.npz"
    fresh(target)
    cases = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    result = np.zeros((len(cases), 4, 3, 224, 224), dtype=np.uint8)
    with np.load(p["cache"], allow_pickle=False) as cache:
        layers = cache["max_layers"]
        for i, patient in cases.iterrows():
            image, _, predicted = _volumes(patient, loc["validation"] / f"{patient.mr}.nii.gz")
            labels = np.asarray(predicted.dataobj)
            z0 = int(layers[int(patient.cache_index)]) - 1
            if z0 < 0 or z0 + 3 >= image.shape[2]:
                raise ValueError("自动掩模四层取层越界。")
            for s in range(4):
                for label in (1, 2, 3):
                    result[i, s, label - 1] = zoom((labels[:, :, z0 + s] == label).astype(np.uint8),
                        (224 / image.shape[0], 224 / image.shape[1]), order=0, prefilter=False)
    np.savez_compressed(target, mrs=cases.mr.to_numpy(dtype=str), masks=result)
    dump_json(loc["root"] / "automatic_input_note.json", {
        "scope": "validation_only", "experiment": "manual_model_auto_mask_substitution",
        "manual_slice_selection_still_required": True,
        "mask_cache_sha256": digest(target),
        "predictions_sha256": {mr: digest(loc["validation"] / f"{mr}.nii.gz") for mr in cases.mr},
        "not_implemented": "用嵌套交叉拟合自动掩模重训分类器的完整自动联合开发流程"})
    return target


def verify_validation_mask_cache(cfg):
    verify_dataset(cfg)
    loc = locations(cfg)
    metadata = json.loads((loc["root"] / "automatic_input_note.json").read_text(encoding="utf-8"))
    if digest(loc["root"] / "validation_masks.npz") != metadata["mask_cache_sha256"]:
        raise ValueError("自动分类掩模缓存发生变化。")
    expected = set(read_cohort(cfg).query("split == 'validation'").mr)
    if set(metadata["predictions_sha256"]) != expected:
        raise ValueError("自动掩模缓存的验证患者不一致。")
    for mr, checksum in metadata["predictions_sha256"].items():
        if digest(loc["validation"] / f"{mr}.nii.gz") != checksum:
            raise ValueError(f"{mr}的原生自动预测与四层缓存不一致，请使用新实验版本。")


## 统一指标、模型比较与绘图

`classification_metrics` 根据每组已确定的阈值计算准确率、敏感度、特异度和混淆矩阵，同时报告 AUC、AUC 重采样区间及 Brier 分数。

`paired_comparison` 按患者编号对齐两个模型，在同一批抽样患者上计算 AUC 差值及区间。`plot_predictions` 绘制 ROC 和校准图：前者观察区分能力，后者把预测概率与实际阳性比例放在一起比较。

`run_validation_report` 将各组预测合并，分别与临床模型比较，并在纳入影像组学时增加 ResNet 与影像组学的配对比较，最后保存指标表、患者预测表和图片。


In [ ]:
"""Classification summaries; thresholds are supplied, never selected on validation."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])


def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result


def paired_comparison(a: pd.DataFrame, b: pd.DataFrame, n_bootstrap=1000):
    for frame in (a, b):
        if not frame["mr"].is_unique or set(frame["label"]) != {0, 1}:
            raise ValueError("比较需要唯一患者编号与两类结局。")
    if set(a["mr"]) != set(b["mr"]):
        raise ValueError("两组患者不完全一致；不静默取交集。")
    merged = a.merge(b, on="mr", suffixes=("_a", "_b"), validate="one_to_one")
    if not (merged.label_a == merged.label_b).all():
        raise ValueError("同一患者的标签不一致。")
    y, pa, pb = merged.label_a.to_numpy(), merged.probability_a.to_numpy(), merged.probability_b.to_numpy()
    if not np.isfinite(pa).all() or not np.isfinite(pb).all():
        raise ValueError("预测包含非有限值。")
    differences = []
    for seed in range(n_bootstrap):
        idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
        differences.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.asarray(differences)
    return {"auc_difference": float(roc_auc_score(y, pa) - roc_auc_score(y, pb)),
            "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)),
            "empirical_p": min(1.0, 2 * min(float((d <= 0).mean()), float((d >= 0).mean())))}


def plot_predictions(predictions):
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, table in predictions.groupby("model"):
        fpr, tpr, _ = roc_curve(table.label, table.probability)
        axes[0].plot(fpr, tpr, label=f"{name}: {roc_auc_score(table.label, table.probability):.3f}")
        truth, prob = calibration_curve(table.label, table.probability, n_bins=5, strategy="quantile")
        axes[1].plot(prob, truth, "o-", label=name)
    for ax in axes:
        ax.plot([0, 1], [0, 1], "k--", alpha=.4)
        ax.legend(fontsize=8)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    axes[1].set(xlabel="Predicted probability", ylabel="Observed proportion", title="Calibration (no refitting)")
    fig.tight_layout()
    return fig


def run_validation_report(cfg, include_radiomics=True, include_automatic=False):
    if cfg.get("data_access") == "classification_cache_only" and (include_radiomics or include_automatic):
        raise ValueError("当前为分类缓存包，请关闭 INCLUDE_RADIOMICS 和 INCLUDE_AUTOMATIC_MASK_SUBSTITUTION。")
    p = paths(cfg)
    fresh(p["reports"] / "validation_predictions.csv", p["reports"] / "validation_metrics.csv")
    predictions, metrics = predict_original_validation(cfg)
    tables, summaries = [predictions], [metrics]
    if include_radiomics:
        predicted, summary = predict_radiomics_validation(cfg)
        tables.append(predicted)
        summaries.append(summary)
    if include_automatic:
        automatic_cache = locations(cfg)["root"] / "validation_masks.npz"
        if not automatic_cache.exists():
            build_validation_mask_cache(cfg)
        verify_validation_mask_cache(cfg)
        predicted, summary = predict_original_validation(cfg, automatic_masks=automatic_cache)
        tables.append(predicted)
        summaries.append(summary)
        evaluate_masks(cfg, scope="validation")
    all_predictions = pd.concat(tables, ignore_index=True)
    all_metrics = pd.concat(summaries, ignore_index=True)
    p["reports"].mkdir(parents=True, exist_ok=True)
    all_predictions.to_csv(p["reports"] / "validation_predictions.csv", index=False)
    all_metrics.to_csv(p["reports"] / "validation_metrics.csv", index=False)
    groups = {name: table for name, table in all_predictions.groupby("model")}
    comparisons = []
    for name in groups:
        if name != "Clinical5":
            comparisons.append({"a": name, "b": "Clinical5", **paired_comparison(groups[name], groups["Clinical5"])})
    if "Radiomics" in groups:
        comparisons.append({"a": "ResNet50", "b": "Radiomics", **paired_comparison(groups["ResNet50"], groups["Radiomics"])})
    pd.DataFrame(comparisons).to_csv(p["reports"] / "paired_auc_comparisons.csv", index=False)
    figure = plot_predictions(all_predictions)
    figure.savefig(p["reports"] / "roc_calibration.png", dpi=150)
    import matplotlib.pyplot as plt
    plt.close(figure)
    dump_json(p["reports"] / "report_notes.json", {
        "resnet": "原结构、训练、单模型及一次OOF晚期融合；没有新增校准或全流程嵌套改造。",
        "radiomics": "新增全三维ROI基线；覆盖范围与四层ResNet不同。" if include_radiomics else "未纳入本次报告。",
        "automatic": "人工掩模训练的原分类器替换输入敏感性实验；仍需人工取层。" if include_automatic else "未纳入本次报告。",
        "auc_p": "经验配对bootstrap；不是DeLong检验。"})
    record(cfg, "validation_report")
    return all_metrics, pd.DataFrame(comparisons)


## 1. 查看模型和输入是否齐备

检查患者清单、分类模型校验清单、影像组学模型清单和可选自动掩模目录是否存在。这里先查看文件准备情况，正式预测时还会逐项核对内容和版本。


In [ ]:
import pandas as pd
readiness = {
    "固定病例": p["cohort"],
    "原ResNet及临床融合锁定清单": p["temporal"] / "frozen_artifacts.json",
    "影像组学锁定清单": p["radiomics"] / "frozen_artifacts.json",
    "自动掩模目录（可选）": p["segmentation"] / "validation_predictions",
}
display(pd.DataFrame([{"item": key, "ready": value.exists(), "path": str(value)} for key, value in readiness.items()]))


## 2. 运行时间验证

默认比较五组模型。影像组学尚未完成时，可将 `INCLUDE_RADIOMICS` 设为 `False`，先生成三组分类基线的结果；准备好 05 的自动分割预测后，可通过 `INCLUDE_AUTOMATIC_MASK_SUBSTITUTION` 加入掩模替换实验。

结果写入当前实验的 `reports/` 目录。若该目录已有同名报告，可直接在下一节读取；重新开展实验时使用新的 `run_id`。


In [ ]:
RUN_VALIDATION = False
INCLUDE_RADIOMICS = True
INCLUDE_AUTOMATIC_MASK_SUBSTITUTION = False

if RUN_VALIDATION:
    metrics, differences = run_validation_report(cfg, include_radiomics=INCLUDE_RADIOMICS,
        include_automatic=INCLUDE_AUTOMATIC_MASK_SUBSTITUTION)
    display(metrics)
    display(differences)


## 3. 查看总体指标与曲线

`validation_metrics.csv` 给出各模型的指标，`paired_auc_comparisons.csv` 给出模型间 AUC 差值和区间。差值为表中 A 模型减 B 模型，可结合区间判断差异估计的精确程度。

ROC 图展示各模型的区分表现；校准图将患者按预测概率分组，比较平均预测概率与实际阳性比例。校准图用于观察概率是否偏高或偏低，此处不会据此重新拟合模型。


In [ ]:
from IPython.display import Image, display
for filename in ["validation_metrics.csv", "paired_auc_comparisons.csv"]:
    target = p["reports"] / filename
    if target.exists():
        print(filename)
        display(pd.read_csv(target))
figure = p["reports"] / "roc_calibration.png"
if figure.exists():
    display(Image(filename=str(figure)))


## 4. 回到患者层面查看误判

读取每位患者的真实标签、预测概率、阈值和最终分类，按模型汇总误判数量，再列出具体病例。可以结合 04 的病例图和临床资料，分析错误是否集中在某些患者特征上。

AUC 描述概率对两类患者的排序能力；准确率、敏感度和特异度描述当前阈值下的分类结果，报告时需要一起看。


In [ ]:
target = p["reports"] / "validation_predictions.csv"
if target.exists():
    predictions = pd.read_csv(target)
    display(predictions.head())
    errors = predictions.loc[predictions.prediction != predictions.label]
    display(errors.groupby("model").size().rename("n_errors"))
    display(errors[["model", "mr", "label", "probability", "threshold", "prediction"]])


## 整理研究结果时需要交代的条件

时间验证结论取决于实际纳入患者、模型版本和本次生成的结果文件。这批验证队列原计划为 61 例，规模较小，既往结果也已被查看；后续若据此反复挑选模型，评价会逐渐偏向这批患者。

方法部分还应说明：ResNet 的开发折参与过最佳轮数选择，融合使用一次生成的影像折外概率；影像组学覆盖完整三维病灶，而 ResNet 使用四层；自动掩模组使用的是人工掩模训练的分类器。

AUC 区间和配对比较基于固定预测的患者重采样，反映病例抽样的不确定性，不包含重新训练模型带来的全部波动。表中的经验 p 值来自配对 bootstrap，统计方法应按实际计算过程注明。
